In [ ]:
# Team: Name1 (ID1), Name2 (ID2), Name3 (ID3)
# Status: setup notebook - runs to the end without error (update if this changes)
# Third-party sources: none yet (add links here if we reuse an existing implementation)

# 00 — Colab setup: Fire & Smoke CNN
Run top to bottom (**Runtime → Run all**) at the start of every Colab session.
Colab wipes `/content` when the runtime disconnects, so the dataset is re-copied each time.

**Before running:** *Runtime → Change runtime type → T4 GPU*.

## 1. Check the GPU and TensorFlow

In [ ]:
import tensorflow as tf                                   # TensorFlow/Keras, as used in the CNN lectures
print("TensorFlow version:", tf.__version__)              # record this in the report for reproducibility
gpus = tf.config.list_physical_devices("GPU")             # list GPUs TensorFlow can see
print("GPUs:", gpus if gpus else "NONE - switch runtime to T4 GPU")  # training on CPU would be far too slow

## 2. Mount Google Drive
The dataset zip lives in the shared Drive folder **CS4287-Project** (each member adds a shortcut to *My Drive*).

In [ ]:
from google.colab import drive                            # Colab helper for Google Drive
drive.mount("/content/drive")                             # asks for permission the first time each session

## 3. Copy the dataset to the local Colab disk
Reading 21k small images straight from Drive is very slow, so we copy **one zip** and unzip it locally.

In [ ]:
import os, glob                                           # filesystem helpers

DRIVE_ZIP  = "/content/drive/MyDrive/CS4287-Project/dfire.zip"  # shared zip on Drive - change if renamed
LOCAL_DATA = "/content/data"                              # fast local disk (wiped when runtime resets)

if not os.path.exists(DRIVE_ZIP):                         # fail early with a clear message
    raise FileNotFoundError(f"Zip not found at {DRIVE_ZIP}. Did you add the shared folder shortcut to My Drive?")

if not os.path.isdir(LOCAL_DATA):                         # only unzip once per session
    !mkdir -p {LOCAL_DATA}
    !cp "{DRIVE_ZIP}" /content/dfire.zip                  # one big copy is much faster than many small reads
    !unzip -q /content/dfire.zip -d {LOCAL_DATA}          # -q = quiet
    !rm /content/dfire.zip                                # free disk space

# The zip may contain Project/train or just train - find the folder that holds train/ and test/
TRAIN_DIR = glob.glob(f"{LOCAL_DATA}/**/train/images", recursive=True)[0].rsplit("/images", 1)[0]
TEST_DIR  = TRAIN_DIR.rsplit("/train", 1)[0] + "/test"
print("Train:", TRAIN_DIR)
print("Test: ", TEST_DIR)

## 4. Sanity-check the dataset and derive image-level labels
Labels are YOLO boxes: `class x y w h` with **0 = smoke, 1 = fire**; an empty file means neither.
For a classifier we turn each label file into one of **none / smoke / fire / both**.

In [ ]:
import pandas as pd                                       # tables for counts

CLASS_NAMES = {0: "smoke", 1: "fire"}                     # D-Fire convention - verify against a few images below

def image_label(label_path):
    """Map a YOLO label file to one image-level class: none / smoke / fire / both."""
    with open(label_path) as f:
        ids = {int(line.split()[0]) for line in f if line.strip()}  # set of class ids present in the image
    if not ids:          return "none"                    # empty file = no fire, no smoke
    if ids == {0}:       return "smoke"
    if ids == {1}:       return "fire"
    return "both"                                         # both 0 and 1 appear

def build_index(split_dir):
    """Return a DataFrame with one row per image: path + derived label."""
    rows = []
    for img in sorted(glob.glob(f"{split_dir}/images/*")):
        stem = os.path.splitext(os.path.basename(img))[0]   # filename without extension
        lbl = f"{split_dir}/labels/{stem}.txt"              # matching label file
        rows.append({"image": img, "label": image_label(lbl) if os.path.exists(lbl) else "MISSING"})
    return pd.DataFrame(rows)

train_df = build_index(TRAIN_DIR)
test_df  = build_index(TEST_DIR)
print(f"train images: {len(train_df)}   test images: {len(test_df)}")
summary = pd.DataFrame({"train": train_df.label.value_counts(), "test": test_df.label.value_counts()}).fillna(0).astype(int)
summary

In [ ]:
import matplotlib.pyplot as plt                           # plotting

summary.plot(kind="bar", figsize=(7, 4))                  # class balance - goes in the Data Set section of the report
plt.title("Image-level class distribution")
plt.ylabel("number of images")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Look at a few examples of each class

In [ ]:
from PIL import Image                                     # image loading

classes = ["none", "smoke", "fire", "both"]
fig, axes = plt.subplots(len(classes), 4, figsize=(12, 10))
for r, c in enumerate(classes):
    sample = train_df[train_df.label == c].sample(4, random_state=42)  # fixed seed so the team sees the same images
    for ax, path in zip(axes[r], sample.image):
        ax.imshow(Image.open(path)); ax.set_title(c); ax.axis("off")
plt.tight_layout(); plt.show()

## 6. Save the index so other notebooks can reuse it

In [ ]:
train_df.to_csv(f"{LOCAL_DATA}/train_index.csv", index=False)   # image path + label per row
test_df.to_csv(f"{LOCAL_DATA}/test_index.csv", index=False)
print("Saved index CSVs to", LOCAL_DATA)

## 7. (Optional) Clone the repo inside Colab to use code from `src/`
Only needed once we have helper files in `src/`. Store a GitHub personal access token in Colab's
**Secrets** panel (key icon on the left) as `GITHUB_TOKEN` — never paste it into the notebook.

In [ ]:
# from google.colab import userdata
# token = userdata.get("GITHUB_TOKEN")
# REPO = "YOUR-GITHUB-USERNAME/fire-smoke-cnn"
# !git clone -q https://{token}@github.com/{REPO}.git /content/repo
# import sys; sys.path.append("/content/repo/src")